# D4GC corpus audit and retrieval baseline

Run top to bottom from a working directory inside this repository. The notebook reads the supplied corpus without changing it. The challenge brief asks for an on premises, fast, cost conscious retrieval tool with few false positives. This notebook audits the supplied text and demonstrates a lexical baseline; it does **not** establish retrieval accuracy without relevance judgments.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Find this repository when the notebook is opened from notebooks/ or the root.
repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'documentation.csv').exists())
sys.path.insert(0, str(repo / 'src'))
from msf_navigator.corpus_analysis import load_corpus, audit, build_index, search
records, metadata = load_corpus()
summary = audit(records, metadata)
summary

## Coverage and metadata

Compare actual files with the metadata table. Missing metadata is retained as `missing`; do not silently infer labels.

In [ ]:
pd.DataFrame(records).groupby(['language', 'genre'], dropna=False).size().sort_values(ascending=False).head(20)

In [ ]:
pd.DataFrame(records)[['id','words','characters','language','genre']].describe(include='all')

## Document quality

Very short text may indicate parsing loss, especially for figures or scans. Exact duplicates are detected by SHA-256; near duplicates are outside this quick audit. Inspect the original PDF/HTML before concluding a short file has no useful content.

In [ ]:
pd.DataFrame(records).nsmallest(15, 'words')[['id','words','language','genre']]

## Offline retrieval baseline

A Unicode word TF-IDF index works without downloaded models. It is language agnostic in implementation, but does not translate queries; test each source language separately. Scores rank results within one query; they are not calibrated probabilities. Reviewers should judge relevance and false positives for real field queries before selecting a production threshold.

In [ ]:
vectorizer, matrix = build_index(records)
print('Index shape:', matrix.shape, 'nonzero values:', matrix.nnz)
for query in ['cholera treatment', 'malnutrition children', 'tuberculosis drug resistance']:
    print('\n', query)
    display(pd.DataFrame(search(query, records, vectorizer, matrix), columns=['id','score','genre','language']))

## Next validation

Create a small, multilingual query set with document relevance labels from domain experts. Report precision at 5, recall at 10, false positives per query, query latency, index size, and performance by language and genre. Compare this baseline with word TF-IDF/BM25 and optional reranking. Keep source IDs in every result so users can open the original document.